In [389]:
# performance testing for metapop1
import pickle
from metapop_value_iteration import _act, build_optimal_controller_fully_observable
from metapop1 import metapop1
import numpy as np
import pandas as pd


def avgperformance(env, config, policy_printout=False):
    policytype = config['policytype']
    num_episodes = config['num_episodes']
    if policytype == 0: # Value iteration
        print( 'policy type: value iteration')
        with open(f'./value_iteration/VI_controller_setting{config["policyid"]}.pkl', 'rb') as f:
            ctrl = pickle.load(f)
        policy = ctrl['policy']
        # reinitiate the environment
        settings = ctrl['envinfo']
        print(settings)
        env = metapop1(settings)
    elif policytype == 1: # ppo
        print( 'policy type: ppo')
        policy = 0
    elif policytype == 2: # heuristics
        if config['heuristics'] == 0:
            print( 'policy type: heuristics, no action')
        elif config['heuristics'] == 1:
            print( 'policy type: heuristics, full action')
    rewards = []
    for i in range(num_episodes):
        obs, state = env.reset()
        if config['POMDP'] == 0:
            input = state
        else:
            input = obs
        done = False
        ep_reward = 0
        while not done:
            if policytype == 0:
                action = _act(env, policy, input)
            elif policytype == 1:
                action = 0
            elif policytype == 2: # heuristics
                if config['heuristics'] == 0: # no action
                    action = np.zeros(env.action_dim)
                elif config['heuristics'] == 1: # full action
                    action = np.ones(env.action_dim)
            if policy_printout:
                print(f't: {input[env.sidx["t"][0]]},  X: {input[env.sidx["X"]]}, aS: {action[env.aidx["aS"]]}, H: {input[env.sidx["H"]]}, aR: {action[env.aidx["aR"]]}')
            obs, reward, done, info = env.step(action)
            if config['POMDP'] == 0:
                input = env.state
            else:
                input = env.obs
            ep_reward += reward
        rewards.append(ep_reward)
        if (i+1) % 100 == 0:
            print(f'Episode {i+1} done')
    print(f'Average reward over {num_episodes} episodes: {np.mean(rewards)}')
    return rewards


config = {'policytype': 0, 'policyid': 1, 'POMDP': 0, 'heuristics': 1, 'num_episodes': 1}
settings = dict(
    partial_observability=0,
    patchnum=4,                 # keep small for exact DP
    action_portfolio=0,
    survey_is_action=0,
    dispersal_regime=1,
    terminal_penalty=1,
    limit_actions=1,
    dispersal_type='uniform',
    dispersal_ID=0,
    paramsetID=1,
)
env = metapop1(settings)

rewards = avgperformance(env,config=config, policy_printout=True)

policy type: value iteration
{'ID': 1, 'partial_observability': 0, 'patchnum': 4, 'action_portfolio': 0, 'survey_is_action': 0, 'dispersal_regime': 1, 'terminal_penalty': 0, 'limit_actions': 1, 'dispersal_type': 'uniform', 'dispersal_ID': 0, 'paramsetID': 1, 'kR': 1, 'kS': 4}
t: 0,  X: [1 1 1 1], aS: [0 0 0 0], H: [1 1 1 1], aR: [0 0 0 0]
t: 1,  X: [1 1 1 1], aS: [0 0 0 0], H: [0 1 1 1], aR: [0 0 0 0]
t: 2,  X: [0 1 1 1], aS: [0 0 0 0], H: [0 1 0 1], aR: [1 0 0 0]
t: 3,  X: [1 1 1 1], aS: [0 0 0 0], H: [1 1 0 1], aR: [0 0 0 0]
t: 4,  X: [1 1 1 1], aS: [0 0 0 0], H: [1 1 0 1], aR: [0 0 0 0]
t: 5,  X: [1 1 1 1], aS: [0 0 0 0], H: [0 1 0 1], aR: [1 0 0 0]
t: 6,  X: [1 1 1 1], aS: [0 0 0 0], H: [1 1 0 1], aR: [0 0 0 0]
t: 7,  X: [1 0 1 1], aS: [0 0 0 0], H: [1 0 0 1], aR: [0 1 0 0]
t: 8,  X: [1 0 0 1], aS: [0 0 0 0], H: [0 1 0 1], aR: [0 0 1 0]
t: 9,  X: [1 1 0 1], aS: [0 0 0 0], H: [0 1 1 1], aR: [0 0 0 0]
t: 10,  X: [1 1 1 1], aS: [0 0 0 0], H: [0 1 1 1], aR: [0 0 0 0]
t: 11,  X: [0 1 1 